In [2]:
import os
import numpy as np
import pandas as pd

# تحديد مسار الملف الأصلي ومسار حفظ الملف النظيف بعد الفلترة
input_path = r"E:\graduation project\data set\archive\combined_dataset.csv"
output_path = r"E:\graduation project\data set\archive\cleaned_iiot_dataset.csv"

# تحديد الـ 8 أعمدة السلوكية (السرعة، الحجم، والوجهات) + عمود التصنيف الرئيسي
feature_columns = [
    "network_packets_all_count",
    "network_packets_src_count",
    "network_packets_dst_count",
    "network_packet-size_avg",
    "network_payload-length_avg",
    "network_time-delta_avg",
    "network_interval-packets",
    "network_ips_dst_count",
]
target_column = "label1"
selected_columns = feature_columns + [target_column]

print("Loading selected 9 columns from dataset...")

# قراءة الـ 9 أعمدة فقط من الهارد لتوفير الرامات وتسريع التنفيذ
df = pd.read_csv(input_path, usecols=selected_columns, low_memory=False)
print("Raw shape:", df.shape)

# استبدال القيم اللانهائية (Infinity) بقيم فارغة (NaN) ثم حذف أي صف ناقص
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# حذف الصفوف المكررة لضمان عدم تحيز الموديل أثناء التدريب
df.drop_duplicates(inplace=True)

# تحويل الأعمدة الرقمية إلى float32 لتقليل حجم الملف واستهلاك الذاكرة للنصف
for col in feature_columns:
  df[col] = pd.to_numeric(df[col], errors="coerce").astype(np.float32)

# التأكد من عدم وجود أي قيم فارغة نتجت عن التحويل الرقمي
df.dropna(inplace=True)

# تنظيف نصوص عمود التصنيف من المسافات الزائدة
df[target_column] = df[target_column].astype(str).str.strip()

print("Cleaned shape:", df.shape)
print("\nClass distribution in label1:")
print(df[target_column].value_counts())

# حفظ النسخة النهائية الخفيفة في نفس الفولدر
df.to_csv(output_path, index=False)
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"\nSaved successfully to: {output_path} ({file_size_mb:.2f} MB)")

Loading selected 9 columns from dataset...
Raw shape: (685671, 9)
Cleaned shape: (430205, 9)

Class distribution in label1:
label1
attack    251710
benign    178495
Name: count, dtype: int64

Saved successfully to: E:\graduation project\data set\archive\cleaned_iiot_dataset.csv (26.55 MB)


In [3]:
import os
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# تحديد مسار الداتا النظيفة ومسار حفظ الموديل النهائي
data_path = r"E:\graduation project\data set\archive\cleaned_iiot_dataset.csv"
model_output_path = (
    r"E:\graduation project\data set\archive\zero_trust_rf_model.joblib"
)

# تحميل الداتا الخفيفة (26.55 ميجا)
df = pd.read_csv(data_path)

# فصل الـ 8 أعمدة السلوكية (X) عن عمود التصنيف (y)
feature_columns = [
    "network_packets_all_count",
    "network_packets_src_count",
    "network_packets_dst_count",
    "network_packet-size_avg",
    "network_payload-length_avg",
    "network_time-delta_avg",
    "network_interval-packets",
    "network_ips_dst_count",
]
X = df[feature_columns]

# تحويل التصنيف النصي إلى أرقام (0 = طبيعي، 1 = هجوم)
y = df["label1"].str.lower().map({"benign": 0, "attack": 1})

# تقسيم البيانات إلى 80% للتدريب و 20% للاختبار مع الحفاظ على نفس نسبة الفئات
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}")

# بناء بايبلاين يجمع التطبيع (StandardScaler) مع خوارزمية Random Forest
# تم تحديد max_depth=16 للحفاظ على حجم ملف صغير وسرعة استجابة بالمايكروثانية على Raspberry Pi
edge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    (
        "rf",
        RandomForestClassifier(
            n_estimators=100,
            max_depth=16,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=42,
        ),
    ),
])

# بدء تدريب الموديل وحساب وقت التدريب
print("\nTraining Random Forest Edge Pipeline...")
train_start = time.time()
edge_pipeline.fit(X_train, y_train)
train_time = time.time() - train_start
print(f"Training completed in: {train_time:.2f} seconds")

# التنبؤ على بيانات الاختبار وقياس متوسط زمن الاستجابة للعينة الواحدة بالمايكروثانية
infer_start = time.perf_counter()
y_pred = edge_pipeline.predict(X_test)
infer_total_sec = time.perf_counter() - infer_start
latency_per_sample_us = (infer_total_sec / len(X_test)) * 1_000_000

# حساب مقاييس الدقة المطلوبة في الفصل الرابع من الكتاب
acc = accuracy_score(y_test, y_pred) * 100
prec = precision_score(y_test, y_pred) * 100
rec = recall_score(y_test, y_pred) * 100
f1 = f1_score(y_test, y_pred) * 100

print("\n--- Model Evaluation Metrics ---")
print(f"Accuracy : {acc:.4f}%")
print(f"Precision: {prec:.4f}%")
print(f"Recall   : {rec:.4f}%")
print(f"F1-Score : {f1:.4f}%")
print(f"Avg Inference Latency: {latency_per_sample_us:.2f} microseconds/sample")

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Detailed Classification Report ---")
print(
    classification_report(
        y_test, y_pred, target_names=["Benign (0)", "Attack (1)"], digits=4
    )
)

# عرض ترتيب أهمية كل عمود في اكتشاف الهجمات
importances = edge_pipeline.named_steps["rf"].feature_importances_
feat_imp = pd.DataFrame(
    {"Feature": feature_columns, "Importance": importances}
).sort_values(by="Importance", ascending=False)
print("\n--- Feature Importance Ranking ---")
print(feat_imp.to_string(index=False))

# حفظ البايبلاين بالكامل (الـ Scaler + الموديل) في ملف واحد مضغوط
joblib.dump(edge_pipeline, model_output_path, compress=3)
model_size_mb = os.path.getsize(model_output_path) / (1024 * 1024)
print(f"\nModel exported successfully to: {model_output_path}")
print(f"Final Edge Model Size: {model_size_mb:.2f} MB")

Training samples: 344164 | Testing samples: 86041

Training Random Forest Edge Pipeline...
Training completed in: 21.13 seconds

--- Model Evaluation Metrics ---
Accuracy : 96.4436%
Precision: 98.6641%
Recall   : 95.2108%
F1-Score : 96.9067%
Avg Inference Latency: 2.84 microseconds/sample

--- Confusion Matrix ---
[[35050   649]
 [ 2411 47931]]

--- Detailed Classification Report ---
              precision    recall  f1-score   support

  Benign (0)     0.9356    0.9818    0.9582     35699
  Attack (1)     0.9866    0.9521    0.9691     50342

    accuracy                         0.9644     86041
   macro avg     0.9611    0.9670    0.9636     86041
weighted avg     0.9655    0.9644    0.9645     86041


--- Feature Importance Ranking ---
                   Feature  Importance
 network_packets_dst_count    0.238716
 network_packets_all_count    0.206883
    network_time-delta_avg    0.164933
network_payload-length_avg    0.100702
   network_packet-size_avg    0.096421
 network_packets

In [4]:
import os
import time
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# تحديد المسارات لحفظ الموديل المحسن والرسومات البيانية الخاصة بالكتاب
base_dir = r"E:\graduation project\data set\archive"
data_path = os.path.join(base_dir, "cleaned_iiot_dataset.csv")
model_output_path = os.path.join(base_dir, "zero_trust_rf_model.joblib")

# تحميل البيانات وتقسيمها
df = pd.read_csv(data_path)
feature_columns = [
    "network_packets_all_count",
    "network_packets_src_count",
    "network_packets_dst_count",
    "network_packet-size_avg",
    "network_payload-length_avg",
    "network_time-delta_avg",
    "network_interval-packets",
    "network_ips_dst_count",
]
X = df[feature_columns]
y = df["label1"].str.lower().map({"benign": 0, "attack": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 1. تدريب النسخة المحسنة من Random Forest لرفع الدقة فوق 98%
rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    (
        "rf",
        RandomForestClassifier(
            n_estimators=100,
            max_depth=24,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=42,
        ),
    ),
])

rf_pipeline.fit(X_train, y_train)
infer_start = time.perf_counter()
y_pred_rf = rf_pipeline.predict(X_test)
rf_latency_us = ((time.perf_counter() - infer_start) / len(X_test)) * 1_000_000
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]

# حفظ الموديل النهائي المحسن
joblib.dump(rf_pipeline, model_output_path, compress=3)
rf_size_mb = os.path.getsize(model_output_path) / (1024 * 1024)

# 2. تدريب Isolation Forest للمقارنة العلمية في الكتاب (Unsupervised Baseline)
scaler_iso = StandardScaler()
X_train_benign = scaler_iso.fit_transform(X_train[y_train == 0])
X_test_scaled = scaler_iso.transform(X_test)

iso_model = IsolationForest(
    n_estimators=100, contamination=0.5, n_jobs=-1, random_state=42
)
iso_model.fit(X_train_benign)
# تحويل مخرجات Isolation Forest (-1 للهجوم، 1 للطبيعي) لتطابق (1 للهجوم، 0 للطبيعي)
y_pred_iso = np.where(iso_model.predict(X_test_scaled) == -1, 1, 0)

# طباعة جدول المقارنة بين الخوارزميتين
print("=== Algorithm Comparison (Chapter 4 Table) ===")
print(
    f"Random Forest    -> Accuracy: {accuracy_score(y_test, y_pred_rf)*100:.2f}% | "
    f"Precision: {precision_score(y_test, y_pred_rf)*100:.2f}% | "
    f"Recall: {recall_score(y_test, y_pred_rf)*100:.2f}% | "
    f"F1: {f1_score(y_test, y_pred_rf)*100:.2f}% | "
    f"Latency: {rf_latency_us:.2f} us | Size: {rf_size_mb:.2f} MB"
)
print(
    f"Isolation Forest -> Accuracy: {accuracy_score(y_test, y_pred_iso)*100:.2f}% | "
    f"Precision: {precision_score(y_test, y_pred_iso)*100:.2f}% | "
    f"Recall: {recall_score(y_test, y_pred_iso)*100:.2f}% | "
    f"F1: {f1_score(y_test, y_pred_iso)*100:.2f}%"
)

# 3. رسم وحفظ مصفوفة الارتباك (Normalized Confusion Matrix)
cm = confusion_matrix(y_test, y_pred_rf, normalize="true")
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt=".3f",
    cmap="Blues",
    xticklabels=["Benign (0)", "Attack (1)"],
    yticklabels=["Benign (0)", "Attack (1)"],
)
plt.title("Normalized Confusion Matrix - Random Forest")
plt.ylabel("Actual Class")
plt.xlabel("Predicted Class")
plt.tight_layout()
plt.savefig(os.path.join(base_dir, "fig_confusion_matrix.png"), dpi=300)
plt.close()

# 4. رسم وحفظ منحنيات ROC و Precision-Recall
fpr, tpr, _ = roc_curve(y_test, y_prob_rf)
roc_auc = auc(fpr, tpr)
prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_prob_rf)
pr_auc = auc(rec_curve, prec_curve)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC AUC = {roc_auc:.4f}")
axes[0].plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--")
axes[0].set_title("ROC Curve")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend(loc="lower right")

axes[1].plot(rec_curve, prec_curve, color="green", lw=2, label=f"PR AUC = {pr_auc:.4f}")
axes[1].set_title("Precision-Recall Curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend(loc="lower left")

plt.tight_layout()
plt.savefig(os.path.join(base_dir, "fig_roc_pr_curves.png"), dpi=300)
plt.close()

# 5. رسم وحفظ ترتيب أهمية الخصائص السلوكية (Feature Importance)
importances = rf_pipeline.named_steps["rf"].feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(8, 5))
plt.barh(
    range(len(indices)),
    importances[indices],
    color="#2b5c8f",
    align="center",
)
plt.yticks(range(len(indices)), [feature_columns[i] for i in indices])
plt.xlabel("Relative Importance")
plt.title("Behavioral Feature Importance (CIC IIoT 2025)")
plt.tight_layout()
plt.savefig(os.path.join(base_dir, "fig_feature_importance.png"), dpi=300)
plt.close()

print("\nAll 3 Chapter-4 figures saved successfully in:", base_dir)

=== Algorithm Comparison (Chapter 4 Table) ===
Random Forest    -> Accuracy: 97.22% | Precision: 98.77% | Recall: 96.45% | F1: 97.59% | Latency: 3.54 us | Size: 22.28 MB
Isolation Forest -> Accuracy: 75.53% | Precision: 72.52% | Recall: 93.70% | F1: 81.76%

All 3 Chapter-4 figures saved successfully in: E:\graduation project\data set\archive


In [5]:
import os
import time
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# تحميل الداتا النظيفة لاختبار أقصى دقة ممكنة
base_dir = r"E:\graduation project\data set\archive"
df = pd.read_csv(os.path.join(base_dir, "cleaned_iiot_dataset.csv"))

feature_columns = [
    "network_packets_all_count",
    "network_packets_src_count",
    "network_packets_dst_count",
    "network_packet-size_avg",
    "network_payload-length_avg",
    "network_time-delta_avg",
    "network_interval-packets",
    "network_ips_dst_count",
]
X = df[feature_columns]
y = df["label1"].str.lower().map({"benign": 0, "attack": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# تجربة عمق مفتوح (max_depth=None) لمعرفة سقف الدقة وحجم الملف الناتج
test_pipe = Pipeline([
    ("scaler", StandardScaler()),
    (
        "rf",
        RandomForestClassifier(
            n_estimators=100,
            max_depth=None,
            min_samples_leaf=1,
            n_jobs=-1,
            random_state=42,
        ),
    ),
])

test_pipe.fit(X_train, y_train)
y_pred = test_pipe.predict(X_test)

temp_path = os.path.join(base_dir, "temp_full_rf.joblib")
joblib.dump(test_pipe, temp_path, compress=3)
size_mb = os.path.getsize(temp_path) / (1024 * 1024)

print(f"Max Accuracy : {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Max Precision: {precision_score(y_test, y_pred)*100:.2f}%")
print(f"Max Recall   : {recall_score(y_test, y_pred)*100:.2f}%")
print(f"Full Model Size: {size_mb:.2f} MB")

Max Accuracy : 96.95%
Max Precision: 98.04%
Max Recall   : 96.73%
Full Model Size: 34.60 MB


In [6]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# تحديد المسارات
base_dir = r"E:\graduation project\data set\archive"
raw_path = os.path.join(base_dir, "combined_dataset.csv")
clean_path = os.path.join(base_dir, "cleaned_iiot_dataset.csv")
model_path = os.path.join(base_dir, "zero_trust_rf_model.joblib")

feature_columns = [
    "network_packets_all_count",
    "network_packets_src_count",
    "network_packets_dst_count",
    "network_packet-size_avg",
    "network_payload-length_avg",
    "network_time-delta_avg",
    "network_interval-packets",
    "network_ips_dst_count",
]

# 1. حساب عدد الصفوف المحذوفة بسبب القيم الفارغة مقابل الصفوف المتكررة
raw_df = pd.read_csv(
    raw_path, usecols=feature_columns + ["label1"], low_memory=False
)
total_raw = len(raw_df)

raw_df.replace([np.inf, -np.inf], np.nan, inplace=True)
after_nan = len(raw_df.dropna())
nan_dropped = total_raw - after_nan

after_dup = len(raw_df.dropna().drop_duplicates())
dup_dropped = after_nan - after_dup

print("=== Data Cleaning Breakdown (Chapter 4) ===")
print(f"Total Raw Records      : {total_raw}")
print(f"Removed (NaN / Inf)    : {nan_dropped}")
print(f"Removed (Duplicates)   : {dup_dropped}")
print(f"Final Unique Records   : {after_dup}")

# 2. مقارنة درجة التدريب بدرجة الاختبار لإثبات عدم وجود Overfitting
df = pd.read_csv(clean_path)
X = df[feature_columns]
y = df["label1"].str.lower().map({"benign": 0, "attack": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = joblib.load(model_path)
train_acc = accuracy_score(y_train, model.predict(X_train)) * 100
test_acc = accuracy_score(y_test, model.predict(X_test)) * 100

print("\n=== Generalization Check (Train vs Test) ===")
print(f"Training Accuracy (80%): {train_acc:.2f}%")
print(f"Testing Accuracy  (20%): {test_acc:.2f}%")
print(f"Generalization Gap     : {abs(train_acc - test_acc):.2f}%")

=== Data Cleaning Breakdown (Chapter 4) ===
Total Raw Records      : 685671
Removed (NaN / Inf)    : 0
Removed (Duplicates)   : 255466
Final Unique Records   : 430205

=== Generalization Check (Train vs Test) ===
Training Accuracy (80%): 98.68%
Testing Accuracy  (20%): 97.22%
Generalization Gap     : 1.46%


In [7]:
import json
import pandas as pd

# تحميل الداتا النظيفة لاستخراج مثالين حقيقيين للتجربة
df = pd.read_csv(
    r"E:\graduation project\data set\archive\cleaned_iiot_dataset.csv"
)

# استخراج أول صف سليم (benign) وأول صف هجوم (attack)
real_benign = df[df["label1"].str.lower() == "benign"].iloc[0]
real_attack = df[df["label1"].str.lower() == "attack"].iloc[0]


def format_for_api(row, device_name):
  return json.dumps(
      {
          "device_id": device_name,
          "network_packets_all_count": float(row["network_packets_all_count"]),
          "network_packets_src_count": float(row["network_packets_src_count"]),
          "network_packets_dst_count": float(row["network_packets_dst_count"]),
          "network_packet_size_avg": float(row["network_packet-size_avg"]),
          "network_payload_length_avg": float(
              row["network_payload-length_avg"]
          ),
          "network_time_delta_avg": float(row["network_time-delta_avg"]),
          "network_interval_packets": float(row["network_interval-packets"]),
          "network_ips_dst_count": float(row["network_ips_dst_count"]),
      },
      indent=2,
  )


print("=== COPY THIS FOR REAL BENIGN (ALLOW) ===")
print(format_for_api(real_benign, "ESP32_Normal_01"))

print("\n=== COPY THIS FOR REAL ATTACK (ISOLATE) ===")
print(format_for_api(real_attack, "ESP32_Hacked_02"))

=== COPY THIS FOR REAL BENIGN (ALLOW) ===
{
  "device_id": "ESP32_Normal_01",
  "network_packets_all_count": 150.0,
  "network_packets_src_count": 67.0,
  "network_packets_dst_count": 83.0,
  "network_packet_size_avg": 236.00667,
  "network_payload_length_avg": 468.5893,
  "network_time_delta_avg": 0.013774226,
  "network_interval_packets": 67.006714,
  "network_ips_dst_count": 49.0
}

=== COPY THIS FOR REAL ATTACK (ISOLATE) ===
{
  "device_id": "ESP32_Hacked_02",
  "network_packets_all_count": 161243.0,
  "network_packets_src_count": 43550.0,
  "network_packets_dst_count": 117693.0,
  "network_packet_size_avg": 60.0,
  "network_payload_length_avg": 0.0,
  "network_time_delta_avg": 7.0246874e-06,
  "network_interval_packets": 0.007032907,
  "network_ips_dst_count": 7.0
}
